# NZZ ContextAI — Explorer Notebook
Setup, Daten laden, Chunking, Upload und Retrieval testen.

## 1. Setup

In [1]:
import sys
sys.path.append('../src')

from config import EMBEDDING_MODEL, RERANKER_MODEL, USE_RERANKING
from preprocess import load_dataset, preprocess
from chunking import chunk_dataframe
from embed import get_supabase_client, load_model, embed_chunks, upload_articles, upload_chunks
from retrieval import retrieve, embed_query, search
from sentence_transformers import CrossEncoder

client   = get_supabase_client()
model    = load_model(EMBEDDING_MODEL)
reranker = CrossEncoder(RERANKER_MODEL) if USE_RERANKING else None

print('Setup abgeschlossen.')
print(f'Reranking aktiv: {USE_RERANKING}')

Lädt embedding model: Qwen/Qwen3-Embedding-0.6B


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Setup abgeschlossen.
Reranking aktiv: False


## 2. Daten laden und vorverarbeiten

In [2]:
train = preprocess(load_dataset('train'))
test  = preprocess(load_dataset('test'))

print(f'Train: {len(train)} Artikel')
print(f'Test:  {len(test)} Artikel')
print()
print('Kategorien:')
print(train['category'].value_counts())
print()
print('Beispiel Titel:', train.iloc[0]['title'])
print('Beispiel Body:', train.iloc[0]['body'][:200])

Train: 9231 Artikel
Test:  1026 Artikel

Kategorien:
category
Panorama         1509
Web              1507
International    1358
Wirtschaft       1269
Sport            1080
Inland            910
Etat              599
Wissenschaft      516
Kultur            483
Name: count, dtype: int64

Beispiel Titel: 21-Jähriger fällt wohl bis Saisonende aus
Beispiel Body: Wien – Rapid muss wohl bis Saisonende auf Offensivspieler Thomas Murg verzichten. Der im Winter aus Ried gekommene 21-Jährige erlitt beim 0:4-Heimdebakel gegen Admira Wacker Mödling am Samstag einen T


## 3. Chunking testen

In [3]:
train_chunks = chunk_dataframe(train)

print(f'Anzahl Chunks: {len(train_chunks)}')
print(f'Durchschnitt Chunks pro Artikel: {len(train_chunks) / len(train):.1f}')
print()
print('Beispiel Chunk:')
print(train_chunks.iloc[0])

Anzahl Chunks: 13400
Durchschnitt Chunks pro Artikel: 1.5

Beispiel Chunk:
article_id                  0bdfe96a-e14b-58b8-8177-6136b8262928
chunk_id                  0bdfe96a-e14b-58b8-8177-6136b8262928-0
chunk_index                                                    0
chunk_text     21-Jähriger fällt wohl bis Saisonende aus. Wie...
title                  21-Jähriger fällt wohl bis Saisonende aus
category                                                   Sport
Name: 0, dtype: object


## 4. Sample hochladen
10 Artikel pro Kategorie in Supabase laden. Vorher Tabellen in DataGrip leeren:
```sql
truncate table chunks, articles;
```

In [4]:
sample       = train.groupby('category').head(10).reset_index(drop=True)
sample_chunks = chunk_dataframe(sample)

print(f'Artikel im Sample: {len(sample)}')
print(f'Chunks im Sample:  {len(sample_chunks)}')
print()
print(sample['category'].value_counts())

Artikel im Sample: 90
Chunks im Sample:  140

category
Sport            10
Kultur           10
Web              10
Wirtschaft       10
Inland           10
Etat             10
International    10
Panorama         10
Wissenschaft     10
Name: count, dtype: int64


In [5]:
embeddings = embed_chunks(model, sample_chunks['chunk_text'].tolist())
upload_articles(client, sample_chunks)
upload_chunks(client, sample_chunks, embeddings)
print('Upload abgeschlossen.')

Generiere Embeddings für 140 Chunks...


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Upload von 140 Artikeln zu Supabase...
90 Artikel hochgeladen
Upload von 140 Chunks zu Supabase...
140 Chunks hochgeladen
Upload abgeschlossen.


## 5. Retrieval testen

In [6]:
queries = [
    'Guardiola und Lewandowski beim HSV',
    'Sundar Pichai Google CEO',
    'Putin Ölproduktion',
    'Österreich EM Qualifikation',
    'Astronaut ISS Raumstation',
]

for query in queries:
    print(f'Query: {query}')
    results = retrieve(query=query, model=model, reranker=reranker, client=client)
    for r in results[:2]:
        score_key = 'rerank_score' if USE_RERANKING else 'similarity'
        print(f"  [{r['category']}] {r['title'][:60]} (Score: {r[score_key]:.2f})")
    print()

Query: Guardiola und Lewandowski beim HSV
  [Sport] Abschiedstournee für Guardiola beginnt beim HSV – Lewandowsk (Score: 0.64)
  [Sport] Österreich bleibt in der EM-Qualifikation für 2016 ungeschla (Score: 0.30)

Query: Sundar Pichai Google CEO
  [Inland] Der frühere Grünen-Chef hält die europafeindliche Haltung de (Score: 0.35)
  [Inland] Der frühere Grünen-Chef hält die europafeindliche Haltung de (Score: 0.35)

Query: Putin Ölproduktion
  [Wirtschaft] Putin: "Einigung, dass wir Menge auf Niveau von Jänner halte (Score: 0.64)
  [Web] Nachrichtenportale wie "Russia Today" und "Sputnik" und beza (Score: 0.33)

Query: Österreich EM Qualifikation
  [Sport] Österreich bleibt in der EM-Qualifikation für 2016 ungeschla (Score: 0.68)
  [Sport] Österreich bleibt in der EM-Qualifikation für 2016 ungeschla (Score: 0.62)

Query: Astronaut ISS Raumstation
  [Etat] Der cineastische TV-Spot soll die Botschaft "Lass dein inner (Score: 0.33)
  [Web] Einfaches, aber geniales Spielprinzip macht Spieler

## 6. Retrieval ohne Reranking (zum Vergleich)

In [7]:
for query in queries:
    print(f'Query: {query}')
    query_embedding = embed_query(model, query)
    results = retrieve(query=query, model=model, reranker=None, client=client, top_k_retrieval=5)
    for r in results[:3]:
        print(f"  [{r['category']}] {r['title'][:60]} (Similarity: {r['similarity']:.3f})")
    print()

Query: Guardiola und Lewandowski beim HSV
  [Sport] Abschiedstournee für Guardiola beginnt beim HSV – Lewandowsk (Similarity: 0.642)
  [Sport] Österreich bleibt in der EM-Qualifikation für 2016 ungeschla (Similarity: 0.299)
  [Sport] Österreich bleibt in der EM-Qualifikation für 2016 ungeschla (Similarity: 0.295)

Query: Sundar Pichai Google CEO
  [Inland] Der frühere Grünen-Chef hält die europafeindliche Haltung de (Similarity: 0.353)
  [Inland] Der frühere Grünen-Chef hält die europafeindliche Haltung de (Similarity: 0.346)
  [Etat] 'Bestandsgarantien für 2017 will Horst Pirker für kein Magaz (Similarity: 0.293)

Query: Putin Ölproduktion
  [Wirtschaft] Putin: "Einigung, dass wir Menge auf Niveau von Jänner halte (Similarity: 0.639)
  [Web] Nachrichtenportale wie "Russia Today" und "Sputnik" und beza (Similarity: 0.327)
  [Etat] 'Kate und Camilla Parker-Bowles würden zanken, seit Stiefsch (Similarity: 0.305)

Query: Österreich EM Qualifikation
  [Sport] Österreich bleibt in der EM-Qu